# 05 反馈循环：TTRL

**场景**：测试时强化学习（TTRL，Zuo et al. 2025）在**没有标签**的测试题上训练模型：每道题采样很多个
答案，多数投票的结果当作伪标签，和多数一致的答案得到奖励，然后更新模型，再去做下一轮。它是一个
"生成 → 打分 → 更新 → 再生成"的循环。

通常这要两套系统：vLLM 负责采样，另一个训练框架负责更新，每一轮把权重从训练端同步回推理端。
在 pllm 里，采样和训练用的是同一个引擎、同一份模型：负载声明一个 LoRA 和它的训练方法，
宿主机上的 Python 循环负责投票和组织训练数据，下一次 `generate` 时一起送进去。

这篇实现 TTRL 最简单的形式：对和多数答案一致的样本做一步监督更新（等价于奖励取 0/1、不减基线的
策略梯度）。在 GSM8K 的 16 道题上跑几轮，观察多数答案的准确率和样本之间的一致程度。最后把同一个循环换成
GRPO：奖励不变，按组内优势给每个样本加权。

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # 只显示 vLLM 的报错，省掉启动日志

import json
import math
import re
import time
from collections import Counter
from dataclasses import dataclass, replace

import torch
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm.host import CausalTrainBatch

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

## 第一步：负载，一个 LoRA 加一个训练方法

- `lora` 是整个实例共享的一份状态（`pllm.InstanceState`），字段 `a`、`b` 是低秩因子。`pllm.Field` 的形状
  函数拿到每个目标投影的输入、输出宽度，所以一个声明就为每层的 q、v 投影各生成一对因子。
- `adapter` 把它作为 LoRA 加到每一层的 q、v 投影上：`y = W x + scale · B A x`。`lora_init` 把 `b` 初始化为零，
  所以训练前模型和原来完全一样；`a` 随机初始化，因为两个因子都是零时梯度也恒为零，LoRA 永远学不动。所以
  pllm 要求带优化器的低秩适配器给出非零的初值（`init=pllm.Computed(...)`）。
- `learn` 带 `@pllm.train(state="lora")`。它的参数来自宿主机送来的一个批次（`pllm.batch(...)`），以及
  `pllm.causal_model("adapter")`：一个装着当前 LoRA 的完整因果语言模型，梯度只流向 LoRA。
  `model.token_logprobs(ids, labels, ...)` 给出每个位置上下一个标签 token 的对数概率，标签为 -100 的位置给 0；
  它分块算 LM head，不需要把整个批次的 logits 一次摊开。方法返回一个标量 loss，框架负责反向和 AdamW。

In [2]:
def lora_init(fields: dict[str, torch.Tensor], generator: torch.Generator) -> None:
    for name, value in fields.items():
        if name.startswith("a_"):
            torch.nn.init.kaiming_uniform_(value, a=math.sqrt(5), generator=generator)
        else:
            value.zero_()


@dataclass(frozen=True)
class TTRL(pllm.Workload):
    rank: int = 8
    learning_rate: float = 1e-4

    @property
    def lora(self) -> pllm.InstanceState:
        return pllm.InstanceState(
            {"a": pllm.Field(lambda inputs, outputs: (self.rank, inputs)),
             "b": pllm.Field(lambda inputs, outputs: (outputs, self.rank))},
            optimizer=pllm.AdamW(self.learning_rate), init=pllm.Computed(lora_init))

    @property
    def adapter(self) -> pllm.model.Adapter:
        targets = (pllm.model.Projection("q_proj"), pllm.model.Projection("v_proj"))  # 每一层
        return pllm.model.Adapter("lora", targets, operator="low_rank", scale=2.0)

    @pllm.train(state="lora")
    def learn(self, input_ids: torch.Tensor = pllm.batch("input_ids"), labels: torch.Tensor = pllm.batch("labels"),
              attention_mask: torch.Tensor = pllm.batch("attention_mask"),
              model: pllm.model.CausalModel = pllm.causal_model("adapter")) -> torch.Tensor:
        logprobs = model.token_logprobs(input_ids, labels, attention_mask=attention_mask)  # [序列, token 数 - 1]
        return -logprobs.sum() / labels[:, 1:].ne(-100).sum()  # 答案 token 上的平均负对数概率

`lora_init` 里的字段名是 `a_0, b_0, a_1, ...`：一个字段在每个目标投影上各有一份，按模型里的顺序编号。

## 第二步：宿主机上的投票

这部分是普通 Python：从 `\boxed{}` 里取答案，统计多数，挑出和多数一致的样本，拼成一个
`CausalTrainBatch`（prompt 部分的标签设为 -100，只在答案上算 loss）。为了控制一次训练的显存，
每轮只取多数票最集中的 8 道题，各取一个和多数一致的样本。

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
with open("gsm8k_test_16.json") as file:  # GSM8K 测试集的前 16 题
    tests = json.load(file)


def normalize(text: str) -> str | None:
    try:
        return f"{float(text.replace(',', '').replace('$', '').strip()):g}"
    except ValueError:
        return None


def boxed(text: str) -> str | None:
    found = re.findall(r"\\boxed\{([^{}]*)\}", text)
    return normalize(found[-1]) if found else None


gold = [normalize(row["a"]) for row in tests]
prompts = [tokenizer.apply_chat_template(
    [{"role": "user", "content": row["q"] + "\nPut the final answer in \\boxed{}."}],
    tokenize=False, add_generation_prompt=True) for row in tests]


def vote(outputs) -> tuple[float, float, list[tuple[list[int], list[int]]]]:
    # 返回：多数答案的准确率、多数票占比的平均值、用于训练的 (prompt, 答案) token 序列。
    correct, agreement, candidates = 0, 0.0, []
    for prompt, output, answer in zip(prompts, outputs, gold):
        answers = [boxed(sample.text) for sample in output.outputs]
        counts = Counter(a for a in answers if a is not None)
        if not counts:
            continue
        majority, count = counts.most_common(1)[0]
        correct += majority == answer
        agreement += count / len(answers)
        sample = next(s for s, a in zip(output.outputs, answers) if a == majority)
        candidates.append((count, tokenizer(prompt, add_special_tokens=False).input_ids, list(sample.token_ids)))
    # 多数票最集中的 8 道题，各取一个和多数一致的样本。
    candidates.sort(key=lambda item: -item[0])
    return correct / len(prompts), agreement / len(prompts), [(p, a) for _, p, a in candidates[:8]]


def training_batch(sequences: list[tuple[list[int], list[int]]]) -> CausalTrainBatch:
    width = max(len(p) + len(a) for p, a in sequences)
    ids = torch.full((len(sequences), width), tokenizer.pad_token_id)
    labels = torch.full_like(ids, -100)
    mask = torch.zeros_like(ids)
    for row, (p, a) in enumerate(sequences):
        ids[row, :len(p) + len(a)] = torch.tensor(p + a)
        labels[row, len(p):len(p) + len(a)] = torch.tensor(a)
        mask[row, :len(p) + len(a)] = 1
    return CausalTrainBatch(ids.cuda(), labels.cuda(), mask.cuda())

## 第三步：闭环

每轮对 16 道题各采样 8 个答案（128 个并发请求），投票，把上一轮挑出的样本作为 `batch=` 送进这一轮的
`generate`：框架在这次调用开始时用它做一步 AdamW 更新，更新后的 LoRA 立刻被后续 token 使用。
`max_host_training_rows` 是一次训练最多接受的序列数，要在 `ExecutionOptions` 里事先声明，框架据此预算显存。

In [4]:
# vLLM 只拿 25% 显存，剩下的留给训练时整模型的前向和反向。
config = pllm.ModelConfig(MODEL, max_model_len=640, max_num_seqs=128, max_num_batched_tokens=2048,
                          gpu_memory_utilization=0.25, seed=0,
                          execution=pllm.ExecutionOptions(max_host_training_rows=8))
sampling = SamplingParams(n=8, temperature=0.8, top_p=0.95, max_tokens=256, seed=0)

with pllm.make_llm(config, workload=TTRL()) as llm:
    batch = None
    for round_index in range(5):
        start = time.perf_counter()
        outputs = llm.generate(prompts, sampling, batch=batch)
        accuracy, agreement, chosen = vote(outputs)
        print(f"第 {round_index} 轮：多数答案准确率 {accuracy:.2f}，多数票占比 {agreement:.2f}，"
              f"挑出 {len(chosen)} 个样本，用时 {time.perf_counter() - start:.0f}s，"
              f"已更新 {llm.diagnostics().update_count} 次")
        batch = training_batch(chosen)

第 0 轮：多数答案准确率 0.31，多数票占比 0.16，挑出 8 个样本，用时 11s，已更新 0 次
第 1 轮：多数答案准确率 0.25，多数票占比 0.16，挑出 8 个样本，用时 22s，已更新 1 次
第 2 轮：多数答案准确率 0.38，多数票占比 0.19，挑出 8 个样本，用时 22s，已更新 2 次
第 3 轮：多数答案准确率 0.38，多数票占比 0.20，挑出 8 个样本，用时 42s，已更新 3 次
第 4 轮：多数答案准确率 0.31，多数票占比 0.23，挑出 8 个样本，用时 43s，已更新 4 次


五轮下来，多数答案的准确率在 0.25 到 0.38 之间起伏，多数票占比从 0.16 升到 0.23。16 道题上这只是一两道题的差别，
**不足以说明效果**：这一篇演示的是闭环怎么搭，不是复现 TTRL 的结论。多数票占比偏低，是因为这个 0.5B 模型
经常不按要求把答案写进 `\boxed{}`，没有答案的题按 0 计。

用时那一列也值得看：有几轮明显更慢（20 到 45 秒对 11 秒），多出来的时间花在训练步的编译上，pllm 遇到
新的批次形状时要重新编译训练图。

## 第四步：换成 GRPO

TTRL 只用和多数一致的样本，权重都是 1。GRPO（Shao et al. 2024）把一道题的一组样本放在一起看：每个样本的
奖励减去组内平均、除以组内标准差，得到它的**优势**；优势为正的样本被推高概率，为负的被压低。奖励仍然是
"和多数答案一致"。

和 TTRL 相比只改两处：

- 训练方法多读一个 `pllm.batch("weights")`：每个序列一个权重，这里就是优势。loss 是每个样本答案 token 的
  平均负对数概率，按优势加权后取平均；
- 宿主机上算出优势，放进 `CausalTrainBatch` 的 `weights` 字段。

`max_host_training_rows=8` 正好放下一道题的 8 个样本，所以每轮挑多数票最集中、但不是全票的那道题（全票时组内
奖励都一样，优势全为零，没有信号）。这里只做最简单的一步在线更新，没有重要性比率的裁剪，也没有 KL 惩罚。

In [5]:
@dataclass(frozen=True)
class GRPO(TTRL):
    @pllm.train(state="lora")
    def learn(self, input_ids: torch.Tensor = pllm.batch("input_ids"), labels: torch.Tensor = pllm.batch("labels"),
              attention_mask: torch.Tensor = pllm.batch("attention_mask"),
              weights: torch.Tensor = pllm.batch("weights"),
              model: pllm.model.CausalModel = pllm.causal_model("adapter")) -> torch.Tensor:
        logprobs = model.token_logprobs(input_ids, labels, attention_mask=attention_mask)
        per_sample = logprobs.sum(-1) / labels[:, 1:].ne(-100).sum(-1)  # 每个样本答案 token 的平均对数概率
        return -(weights * per_sample).mean()


def grpo_group(outputs) -> CausalTrainBatch:
    """多数票最集中、但不是全票的那道题：它的全部样本，权重是组内归一化的优势。"""
    groups = []
    for prompt, output in zip(prompts, outputs):
        answers = [boxed(sample.text) for sample in output.outputs]
        counts = Counter(a for a in answers if a is not None)
        if not counts:
            continue
        majority, count = counts.most_common(1)[0]
        if count == len(answers):  # 全票：组内奖励都一样，没有信号
            continue
        rewards = torch.tensor([float(a == majority) for a in answers])
        ids = tokenizer(prompt, add_special_tokens=False).input_ids
        groups.append((count, [(ids, list(sample.token_ids)) for sample in output.outputs],
                       (rewards - rewards.mean()) / rewards.std(correction=0)))
    _, sequences, advantages = max(groups, key=lambda group: group[0])
    return replace(training_batch(sequences), weights=advantages.cuda())

In [6]:
with pllm.make_llm(config, workload=GRPO()) as llm:
    batch = None
    for round_index in range(5):
        start = time.perf_counter()
        outputs = llm.generate(prompts, sampling, batch=batch)
        accuracy, agreement, _ = vote(outputs)
        print(f"第 {round_index} 轮：多数答案准确率 {accuracy:.2f}，多数票占比 {agreement:.2f}，"
              f"用时 {time.perf_counter() - start:.0f}s，已更新 {llm.diagnostics().update_count} 次")
        batch = grpo_group(outputs)

第 0 轮：多数答案准确率 0.31，多数票占比 0.16，用时 12s，已更新 0 次
第 1 轮：多数答案准确率 0.38，多数票占比 0.16，用时 22s，已更新 1 次
第 2 轮：多数答案准确率 0.38，多数票占比 0.17，用时 11s，已更新 2 次
第 3 轮：多数答案准确率 0.31，多数票占比 0.19，用时 23s，已更新 3 次
第 4 轮：多数答案准确率 0.31，多数票占比 0.18，用时 23s，已更新 4 次


GRPO 的五轮里，多数答案准确率在 0.31 和 0.38 之间，多数票占比在 0.16 到 0.19 之间。每轮只用一道题的 8 个样本
更新一次，和 TTRL 那一节一样，这些数字只说明循环跑通了，不说明效果。

## 小结

- 训练方法读宿主机批次（`pllm.batch(...)`）和带 LoRA 的整模型（`pllm.causal_model(...)`），返回一个标量 loss；
- 投票、奖励、挑样本都是宿主机上的普通 Python，结果通过下一次 `generate(batch=...)` 送进引擎；
- 带优势的更新（GRPO）只是批次多一个逐序列的 `weights` 字段、loss 里多乘一个权重；
- 采样和训练共用一个引擎、一份权重，没有权重同步。

pllm 也有更完整的宿主反馈接口：`llm.generate_feedback(...)` 每轮生成后调用你的反馈函数，按它的决定重试部分
请求，并管理投票组。这一篇没有用它，因为这里每轮都要重新生成全部题目，并把训练批次通过 `generate(batch=...)`
交给 LoRA；`generate_feedback` 只重试被标记的请求，两轮之间也只能送入 `StateImport`。内置的 `pllm.workloads.RewardHeadFeedback` 就是这样一个反馈函数：它对每道题的
`\boxed{}` 答案做多数投票，再把一个奖励头（`pllm.workloads.RewardHeadWorkload`）拟合到投票得出的 0/1 奖励上，
让它从每个回答的最终 hidden 预测奖励。奖励头只在宿主上训练，不需要模型：`pllm.host.make_host(config, workload)`
返回一个 `HostTrainer`，每次 `train(batch)` 走一步。

下一篇 [06 读模型内部](06_reading_inside.ipynb) 换一个方向：不训练，只读模型内部的残差流和逐层 logits。